In [ ]:
# ================== LiNGAM + DoWhy (By Region with Predefined Top-6 Features) =====
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from lingam import DirectLiNGAM

# -------- 配置 --------
DATA_PATH = r"D:\研究\ABF新\8.13ML\ML\生产率及解释变量合并2.xlsx"
SAVE_DIR = r"D:\研究\ABF新\因果推断"
os.makedirs(SAVE_DIR, exist_ok=True)

TARGET = "kcal_per_kg"  # 或 "gProtein_per_kg"
REGION_COL = "Region"
RANDOM_STATE = 42

# winsorize 配置
WINS_P_LOW, WINS_P_HIGH = 0.01, 0.99
WINSORIZE_CONTROLS = True

# ----- DoWhy 检查 -----
try:
    from dowhy import CausalModel
    DOWHY_AVAILABLE = True
except Exception:
    DOWHY_AVAILABLE = False
    print("[WARN] DoWhy 不可用，将以 OLS 兜底")

# ----------------- 各地区前6重要特征配置 -----------------
REGION_TOP_FEATURES = {
    "SAS": ["Precipitation", "Import_Dependence_Ratio", "Stocking_Density", "GDP", "Feed_Yield", "Export_Orientation"],
    "SSA": ["Poultry_Share", "Slaughter_Efficiency", "Ruminant_Share", "Precipitation", "SSR", "ABF_Consumption_Percapita"],
    "MNA": ["Ruminant_Share", "Temperature", "GDP", "Precipitation", "Stocking_Density", "SSR"],
    "SEA": ["Slaughter_Efficiency", "Poultry_Share", "Feed_Yield", "Precipitation", "GDP", "Ruminant_Share"],
    "LAM": ["GDP", "Poultry_Share", "SSR", "Feed_Yield", "ABF_Consumption_Percapita", "Precipitation"],
    "EAS": ["Feed_Yield", "SSR", "ABF_Consumption_Percapita", "Precipitation", "Import_Dependence_Ratio", "Export_Orientation"],
    "NAM": ["GDP", "Poultry_Share", "Import_Dependence_Ratio", "Ruminant_Share", "Feed_Yield", "Export_Orientation"],
    "EUR": ["ABF_Consumption_Percapita", "GDP", "Ruminant_Share", "Poultry_Share", "Precipitation", "Temperature"],
    "OCE": ["Stocking_Density", "Precipitation", "SSR", "ABF_Consumption_Percapita", "GDP", "Feed_Yield"]
}

# 特征列名映射（显示名 -> 实际列名）
FEATURE_NAME_MAPPING = {
    "Precipitation": "Pr",
    "Import Dependence Ratio": "Import_Dependence_Ratio", 
    "Stocking Density": "Stocking_Density",
    "GDP": "GDP",
    "Feed Yield": "Feed_Yield",
    "Export Orientation": "Export_Orientation",
    "Poultry Share": "Poultry_Share",
    "Slaughter Efficiency": "Slaughter_Efficiency",
    "Ruminant Share": "Ruminant_Share",
    "SSR": "SSR",
    "ABF Consumption Per Capita": "ABF_Consumption_Percapita",
    "Temperature": "T"
}

# ----------------- 工具函数 -----------------
def winsorize_series(s: pd.Series, p_low=0.01, p_high=0.99):
    s = pd.to_numeric(s, errors="coerce")
    lo, hi = s.quantile(p_low), s.quantile(p_high)
    return s.clip(lower=lo, upper=hi)

def zscore_arr(a):
    a = np.asarray(a, dtype=float)
    mu = np.nanmean(a, axis=0)
    sd = np.nanstd(a, axis=0, ddof=0) + 1e-12
    return (a - mu) / sd

def run_lingam_analysis(df_region, features, target, output_prefix):
    """运行LiNGAM分析并保存结果"""
    use_cols = features + [target]
    data = df_region[use_cols].dropna()
    
    if data.shape[0] < 50:
        print(f"[Skip] {output_prefix}: 样本量{data.shape[0]} < 50")
        return []

    # LiNGAM模型
    model = DirectLiNGAM(random_state=RANDOM_STATE)
    model.fit(data.values)
    adjacency_matrix = model.adjacency_matrix_

    # 保存邻接矩阵
    pd.DataFrame(adjacency_matrix, index=use_cols, columns=use_cols)\
      .to_csv(f"{output_prefix}_lingam_adj_matrix.csv", encoding="utf-8-sig")

    # 提取边信息
    edges = []
    for i, source in enumerate(use_cols):
        for j, target_var in enumerate(use_cols):
            if i != j and abs(adjacency_matrix[i, j]) > 1e-8:
                edges.append((source, target_var, float(adjacency_matrix[i, j])))
    
    edges_df = pd.DataFrame(edges, columns=["source", "target", "coef"])
    edges_df.to_csv(f"{output_prefix}_lingam_edges.csv", index=False, encoding="utf-8-sig")

    # 提取目标变量的父节点
    parents = edges_df[edges_df["target"] == target].copy()
    if parents.empty:
        pd.DataFrame(columns=["source", "target", "coef"]).to_csv(
            f"{output_prefix}_lingam_parents.csv", index=False, encoding="utf-8-sig")
        return []
    
    parents["abs_coef"] = parents["coef"].abs()
    parents = parents.sort_values("abs_coef", ascending=False)
    parents[["source", "target", "coef"]].to_csv(
        f"{output_prefix}_lingam_parents.csv", index=False, encoding="utf-8-sig")
    
    return parents["source"].tolist()

def estimate_causal_effect(df_region, treatment, outcome, common_causes, output_prefix):
    """估计因果效应并进行稳健性检验"""
    cols = [outcome, treatment] + list(common_causes)
    data_subset = df_region[cols].apply(pd.to_numeric, errors="coerce").dropna().copy()
    
    if data_subset.shape[0] < 50:
        return {
            "treatment": treatment, "ATE": np.nan, "ATE_per_1pct": np.nan, 
            "std_ATE": np.nan, "PT": np.nan, "RCC": np.nan, "DS": np.nan, 
            "n": data_subset.shape[0]
        }

    # Winsorize处理
    data_subset[treatment] = winsorize_series(data_subset[treatment], WINS_P_LOW, WINS_P_HIGH)
    if WINSORIZE_CONTROLS:
        for cause in common_causes:
            data_subset[cause] = winsorize_series(data_subset[cause], WINS_P_LOW, WINS_P_HIGH)

    # 1) ATE估计
    if DOWHY_AVAILABLE:
        causal_model = CausalModel(
            data=data_subset,
            treatment=treatment,
            outcome=outcome,
            common_causes=list(common_causes) if common_causes else None
        )
        
        identified_estimand = causal_model.identify_effect(proceed_when_unidentifiable=True)
        estimate = causal_model.estimate_effect(identified_estimand, method_name="backdoor.linear_regression")
        ate = float(estimate.value)
        
        # 稳健性检验
        def run_refutation(method_name):
            try:
                ref_result = causal_model.refute_estimate(identified_estimand, estimate, method_name=method_name)
                return float(ref_result.new_effect)
            except Exception:
                return np.nan
        
        placebo_effect = run_refutation("placebo_treatment_refuter")
        random_cause_effect = run_refutation("random_common_cause")
        data_subset_effect = run_refutation("data_subset_refuter")
        
        # 保存详细报告
        with open(f"{output_prefix}_{treatment}_dowhy_report.txt", "w", encoding="utf-8") as f:
            f.write(str(estimate))
    else:
        # OLS备选方案
        X = data_subset[[treatment] + list(common_causes)].values.astype(float)
        y = data_subset[outcome].values.astype(float)
        ate = float(LinearRegression().fit(X, y).coef_[0])
        
        rng = np.random.default_rng(RANDOM_STATE)
        
        # 安慰剂检验
        X_placebo = X.copy()
        treatment_shuffled = X_placebo[:, 0].copy()
        rng.shuffle(treatment_shuffled)
        X_placebo[:, 0] = treatment_shuffled
        placebo_effect = float(LinearRegression().fit(X_placebo, y).coef_[0])
        
        # 随机共同原因
        X_random_cause = np.column_stack([X, rng.normal(size=X.shape[0])])
        random_cause_effect = float(LinearRegression().fit(X_random_cause, y).coef_[0])
        
        # 数据子集
        subset_idx = rng.choice(np.arange(X.shape[0]), size=X.shape[0]//2, replace=False)
        data_subset_effect = float(LinearRegression().fit(X[subset_idx], y[subset_idx]).coef_[0])

    # 2) 标准化ATE
    X_std = data_subset[[treatment] + list(common_causes)].values.astype(float)
    y_std = data_subset[outcome].values.astype(float)
    X_std = zscore_arr(X_std)
    y_std = zscore_arr(y_std)
    std_ate = float(LinearRegression().fit(X_std, y_std).coef_[0])

    # 3) 百分比效应（如果treatment是比例变量）
    treatment_min, treatment_max = float(data_subset[treatment].min()), float(data_subset[treatment].max())
    ate_per_1pct = ate * 0.01 if (treatment_min >= 0.0 and treatment_max <= 1.0) else np.nan

    return {
        "treatment": treatment, "ATE": ate, "ATE_per_1pct": ate_per_1pct,
        "std_ATE": std_ate, "PT": placebo_effect, "RCC": random_cause_effect, 
        "DS": data_subset_effect, "n": int(data_subset.shape[0])
    }

# ----------------- 主执行流程 -----------------
print("开始分地区因果推断分析...")
print(f"数据路径: {DATA_PATH}")
print(f"目标变量: {TARGET}")

# 加载数据
try:
    df = pd.read_excel(DATA_PATH)
    print(f"数据加载成功，形状: {df.shape}")
except Exception as e:
    print(f"数据加载失败: {e}")
    exit()

if REGION_COL not in df.columns:
    print(f"错误: 数据框中不存在区域列 '{REGION_COL}'")
else:
    regions = sorted(df[REGION_COL].dropna().unique().tolist())
    print(f"发现区域: {regions}")
    
    all_region_results = []
    
    for region in regions:
        print(f"\n处理区域: {region}")
        region_data = df[df[REGION_COL] == region].copy()
        
        if region not in REGION_TOP_FEATURES:
            print(f"  跳过: 未配置该区域的前6特征")
            continue
            
        if region_data.shape[0] < 50:
            print(f"  跳过: 样本量{region_data.shape[0]} < 50")
            continue
        
        # 获取该区域前6重要特征
        region_top_features = REGION_TOP_FEATURES[region]
        
        # 检查特征是否存在
        missing_features = [f for f in region_top_features if f not in df.columns]
        if missing_features:
            print(f"  警告: 以下特征不存在: {missing_features}")
            region_top_features = [f for f in region_top_features if f in df.columns]
        
        if not region_top_features:
            print(f"  跳过: 无可用特征")
            continue
            
        print(f"  前6重要特征: {region_top_features}")
        print(f"  样本量: {region_data.shape[0]}")
        
        # 创建区域输出目录
        region_dir = os.path.join(SAVE_DIR, f"region_{region}")
        os.makedirs(region_dir, exist_ok=True)
        
        region_prefix = os.path.join(region_dir, f"{region}_{TARGET}")
        
        # LiNGAM分析
        lingam_parents = run_lingam_analysis(region_data, region_top_features, TARGET, region_prefix)
        treatment_vars = lingam_parents if lingam_parents else region_top_features
        
        print(f"  LiNGAM发现的父节点: {lingam_parents}")
        
        # 因果效应估计
        for treatment in treatment_vars:
            common_causes = [feature for feature in region_top_features if feature != treatment]
            
            print(f"  估计 {treatment} → {TARGET} 的因果效应...")
            result = estimate_causal_effect(
                region_data, treatment, TARGET, common_causes, region_prefix
            )
            
            result.update({
                "region": region,
                "target": TARGET,
                "top_features": ",".join(region_top_features),
                "lingam_parents": ",".join(lingam_parents)
            })
            
            all_region_results.append(result)
            print(f"    ATE: {result['ATE']:.6f}, std_ATE: {result['std_ATE']:.6f}, 样本量: {result['n']}")
    
    # 保存所有结果
    if all_region_results:
        results_df = pd.DataFrame(all_region_results)
        results_df = results_df[[
            "region", "target", "treatment", "ATE", "ATE_per_1pct", "std_ATE",
            "PT", "RCC", "DS", "n", "top_features", "lingam_parents"
        ]]
        
        output_path = os.path.join(SAVE_DIR, "regional_causal_analysis_results.csv")
        results_df.to_csv(output_path, index=False, encoding="utf-8-sig")
        print(f"\n分析完成! 结果已保存至: {output_path}")
        
        # 显示汇总统计
        valid_results = results_df[results_df['ATE'].notna()]
        print(f"\n汇总统计:")
        print(f"有效因果效应估计数: {len(valid_results)}")
        print(f"涉及区域数: {valid_results['region'].nunique()}")
        print(f"涉及变量数: {valid_results['treatment'].nunique()}")
    else:
        print("\n警告: 未生成任何有效结果")

print(f"\n所有处理完成! 输出目录: {SAVE_DIR}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import PartialDependenceDisplay
from pygam import LinearGAM, s
from matplotlib.cm import ScalarMappable
import matplotlib as mpl
from sklearn.linear_model import LinearRegression

# =======================
# 1. 读数据
# =======================
file_path = r"D:\研究\ABF新\8.13ML\ML\生产率及解释变量合并2.xlsx"
df = pd.read_excel(file_path, sheet_name=0)
y_col = "gProtein_per_kg"
# 调整顺序：Feed_Yield, Ruminant_Share, T
x_cols = ["Feed_Yield", "Ruminant_Share", "T"]

# 只保留需要的列并去掉 NaN
data = df[x_cols + [y_col]].dropna().copy()

# =======================
# 2. IQR 异常值剔除
# =======================
def iqr_mask(series, k=1.5):
    """根据 IQR 返回一个布尔掩码：True = 保留, False = 异常值"""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return (series >= lower) & (series <= upper)

mask = np.ones(len(data), dtype=bool)
for col in x_cols + [y_col]:
    mask &= iqr_mask(data[col])
data_clean = data.loc[mask].copy()
print(f"原始样本数: {len(data)}, 去除 IQR 异常值后样本数: {len(data_clean)}")

X = data_clean[x_cols]
y = data_clean[y_col].values

# =======================
# 3. 训练 PDP 用的模型
# =======================
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X, y)

# =======================
# 4. 全局绘图参数（增大字号）
# =======================
plt.rcParams["font.sans-serif"] = ["SimHei", "Arial"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.size"] = 16
plt.rcParams["axes.titlesize"] = 18
plt.rcParams["axes.labelsize"] = 16
plt.rcParams["xtick.labelsize"] = 14
plt.rcParams["ytick.labelsize"] = 14
plt.rcParams["legend.fontsize"] = 14

# 变量标签映射
x_labels = {
    "Feed_Yield": "Feed Yield",
    "Ruminant_Share": "Ruminant%",
    "T": "T"
}
y_label = "gProtein/kg"

# =======================
# 5. 创建子图布局
# =======================
fig = plt.figure(figsize=(12, 8))
gs = fig.add_gridspec(3, 3, height_ratios=[1, 1, 0.08], hspace=0.35, wspace=0.25)

# 上两行用于子图 a–f
axes = []
for i in range(6):
    row = i // 3
    col = i % 3
    axes.append(fig.add_subplot(gs[row, col]))

# 第三行用于色条（占满整行）
cbar_ax1 = fig.add_subplot(gs[2, 0])
cbar_ax2 = fig.add_subplot(gs[2, 1])
cbar_ax3 = fig.add_subplot(gs[2, 2])

# 子图标签
subplot_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]

# =======================
# 6. 顶行：a, b, c 响应曲线（GAM 曲线 + 置信区间 + 线性趋势线）
# =======================
for i, var in enumerate(x_cols):
    ax = axes[i]
    x = data_clean[var].values

    # (1) GAM 拟合线（基于完整样本）
    gam = LinearGAM(s(0)).fit(x.reshape(-1, 1), y)
    xx = np.linspace(np.nanpercentile(x, 2), np.nanpercentile(x, 98), 500)
    yy = gam.predict(xx.reshape(-1, 1))
    
    # 获取置信区间
    confi = gam.confidence_intervals(xx.reshape(-1, 1), width=0.95)
    
    # 绘制置信区间（使用散点同款颜色 #90B4CF）
    ax.fill_between(xx, confi[:, 0], confi[:, 1], 
                     color="#90B4CF", alpha=0.4, 
                     label="95% CI")

    # (2) GAM 曲线
    gam_line = ax.plot(
        xx, yy, linestyle="-", linewidth=2.5,
        color="#337BAC", label="GAM curve"
    )

    # (3) 添加线性趋势线
    # 使用原始数据（不采样）进行线性拟合
    x_linear = x.reshape(-1, 1)
    y_linear = y
    
    lin_reg = LinearRegression()
    lin_reg.fit(x_linear, y_linear)
    
    # 生成线性预测值
    x_range = np.linspace(x.min(), x.max(), 2)  # 只需要起点和终点
    y_linear_pred = lin_reg.predict(x_range.reshape(-1, 1))
    
    # 绘制线性趋势线
    linear_line = ax.plot(
        x_range, y_linear_pred, 
        linestyle="--", linewidth=2.0,
        color="#FF6B6B", label="Linear trend"
    )

    # 坐标轴标签（使用修改后的标签）
    ax.set_xlabel(x_labels.get(var, var))
    ax.set_ylabel(y_label)

    # 只在 (a) 子图中添加图例
    if i == 0:
        ax.legend(frameon=True, loc='upper right')

    # 添加子图编号
    ax.text(
        0.02, 0.98, subplot_labels[i],
        transform=ax.transAxes,
        fontsize=14, fontweight='bold',
        va='top', ha='left'
    )

    # 调整刻度字号
    ax.tick_params(axis='both', labelsize=14)

# =======================
# 7. 底行：d, e, f PDP 交互作用（去掉标题 & 去掉下划线）
# =======================
pdp_plots = []

pdp_pairs = [
    ("Feed_Yield", "Ruminant_Share"),
    ("Feed_Yield", "T"),
    ("Ruminant_Share", "T")
]

for i, (feat1, feat2) in enumerate(pdp_pairs):
    # 先把 axes[i+3] 交给 PDP 画
    display = PartialDependenceDisplay.from_estimator(
        rf, X, [(feat1, feat2)],
        ax=axes[i + 3], kind="average"
    )
    pdp_plots.append(display)

    # 从 display 中拿真正的轴
    if hasattr(display, "axes_"):
        pdp_ax = np.ravel(display.axes_)[0]
    else:
        pdp_ax = axes[i + 3]

    # 改成无下划线的标签
    pdp_ax.set_xlabel(x_labels[feat1])
    pdp_ax.set_ylabel(x_labels[feat2])

    # 去掉自动生成的标题
    pdp_ax.set_title("")

    # 子图编号
    pdp_ax.text(
        0.02, 0.98, subplot_labels[i + 3],
        transform=pdp_ax.transAxes,
        fontsize=14, fontweight='bold',
        va='top', ha='left'
    )

    pdp_ax.tick_params(axis='both', labelsize=11)

# =======================
# 8. 添加 PDP 色条（并调窄色带厚度）
# =======================
for i, (display, cax) in enumerate(zip(pdp_plots, [cbar_ax1, cbar_ax2, cbar_ax3])):
    # 从 PDP 图中提取数据
    pdp_results = display.pd_results[0]
    pdp_values = pdp_results["average"]

    # 创建色条
    norm = mpl.colors.Normalize(vmin=pdp_values.min(), vmax=pdp_values.max())
    sm = ScalarMappable(norm=norm, cmap='viridis')
    sm.set_array([])

    cbar = plt.colorbar(sm, cax=cax, orientation='horizontal')
    cbar.set_label('Partial dependence', fontsize=14)
    cbar.ax.tick_params(labelsize=10)

# 先整体布局，再手动压缩色带厚度（长度保持不变）
plt.tight_layout()

for cax in [cbar_ax1, cbar_ax2, cbar_ax3]:
    pos = cax.get_position()
    new_height = pos.height * 0.5  # 厚度减半
    # 调整色带与图的距离：增加0.02让色带离图更远
    new_y0 = pos.y0 + 0.02
    cax.set_position([pos.x0, new_y0, pos.width, new_height])


# 先保存
plt.savefig(
    r"D:\研究\ABF新\论文的图\response_PDP.png",
    dpi=300,
    bbox_inches="tight"
)

# 再显示
plt.show()